# Background Removal (RVM)

إزالة الخلفية باستخدام Robust Video Matting (P4-T3) على Kaggle GPU.

المميزات:
- قناع شفافية بجودة سينمائية بلا هالة حول الشعر
- 5 خلفيات احترافية جاهزة (عيادة، تدرّج طبي، ستوديو أبيض، إنفوجرافيك، دريمي)
- color match بين كل الفريمات
- رفع النتائج إلى GitHub Release

In [ ]:
# Cell 2 — تثبيت RVM + torch + opencv
import subprocess, os
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q opencv-python Pillow moviepy
!apt-get update -qq && apt-get install -y -qq ffmpeg

# Clone RVM repo (model weights auto-downloaded on first run)
!git clone --depth 1 https://github.com/PeterL1n/RobustVideoMatting.git /kaggle/working/rvm 2>/dev/null || true
import sys
sys.path.insert(0, '/kaggle/working/rvm')

import torch, cv2, numpy as np
print(f'CUDA: {torch.cuda.is_available()}, GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

In [ ]:
# Cell 3 — تحميل نموذج RVM
import os, time, torch
sys.path.insert(0, '/kaggle/working/rvm')
try:
    from model import MattingNetwork
    from inference_utils import VideoReader, VideoWriter
except Exception as e:
    print(f'[WARN] Could not import RVM modules locally ({e})')
    print('       Trying pip install of community wheel ...')
    !pip install -q robust-video-matting
    from rvm.model import MattingNetwork
    from rvm.inference_utils import VideoReader, VideoWriter

MODEL_VARIANT = os.environ.get('RVM_MODEL', 'resnet50')  # mobilenetv3 | resnet50
print(f'Loading RVM ({MODEL_VARIANT}) ...')
t0 = time.time()
model = MattingNetwork(MODEL_VARIANT).eval().cuda()

# Auto-download checkpoint
ckpt_path = f'/kaggle/working/rvm_{MODEL_VARIANT}.pth'
if not os.path.exists(ckpt_path):
    url = f'https://github.com/PeterL1n/RobustVideoMatting/releases/download/v1.0.0/rvm_{MODEL_VARIANT}.pth'
    !wget -q {url} -O {ckpt_path}
model.load_state_dict(torch.load(ckpt_path, map_location='cuda'))
print(f'[OK] RVM loaded in {time.time()-t0:.1f}s')

In [ ]:
# Cell 4 — معالجة الفيديو فريم-بفريم
import os, time, torch
import numpy as np, cv2
from inference_utils import VideoReader, VideoWriter

input_video = os.environ.get('INPUT_VIDEO', '/kaggle/input/test_video.mp4')
if not os.path.exists(input_video):
    raise FileNotFoundError(f'Input video not found: {input_video}')

out_dir = '/kaggle/working/output'
os.makedirs(out_dir, exist_ok=True)
matte_path = os.path.join(out_dir, 'matte_composited.mp4')
alpha_path = os.path.join(out_dir, 'alpha_mask.mp4')

# Pick a background (default dental_office per P4-T3)
BG_NAME = os.environ.get('RVM_BACKGROUND', 'dental_office')
ASSETS_DIR = os.environ.get('MICKY_BGS_DIR', '/kaggle/input/micky-assets/backgrounds')
bg_path = os.path.join(ASSETS_DIR, f'{BG_NAME}.png')

reader = VideoReader(input_video, output_type='numpy')
writer_alpha = VideoWriter(alpha_path, fps=reader.fps)
writer_comp  = VideoWriter(matte_path, fps=reader.fps)

# Load background image (or use solid color fallback)
bgr = None
if os.path.exists(bg_path):
    bgr_img = cv2.imread(bg_path, cv2.IMREAD_COLOR)
    print(f'Using background: {bg_path}')
else:
    print(f'[WARN] Background image not found ({bg_path}); using dark blue gradient')
    bgr_img = np.zeros((1080, 1920, 3), dtype=np.uint8)
    for i in range(1080):
        bgr_img[i, :] = [10 + i*0.02, 20 + i*0.03, 40 + i*0.04]

bgr_tensor = None  # built lazily on first frame so size matches

with torch.no_grad():
    rec = [None] * 4  # recurrent states for RVM
    n = len(reader)
    print(f'Processing {n} frames ...')
    t0 = time.time()
    for i, frame in enumerate(reader):  # frame: HxWx3 RGB uint8
        src = torch.from_numpy(frame).permute(2, 0, 1).unsqueeze(0).float().cuda() / 255.0
        fgr, pha, *rec = model(src, *rec)
        # pha: 1x1xHxW alpha mask
        alpha_np = (pha[0, 0].cpu().numpy() * 255).astype(np.uint8)
        writer_alpha.write_cv(alpha_np)

        # Composite: foreground over background (per P4-T3)
        if bgr_tensor is None or bgr_tensor.shape != src.shape:
            h, w = src.shape[-2:]
            bg_resized = cv2.resize(bgr_img, (w, h))
            bgr_tensor = torch.from_numpy(bg_resized).permute(2, 0, 1).unsqueeze(0).float().cuda() / 255.0
        composited = fgr * pha + bgr_tensor * (1 - pha)
        comp_np = (composited[0].permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
        writer_comp.write_cv(comp_np[:, :, ::-1])  # BGR for OpenCV writer

        if (i + 1) % 30 == 0 or i == n - 1:
            print(f'  frame {i+1}/{n}  ({(i+1)/(time.time()-t0):.1f} fps)')

writer_alpha.close()
writer_comp.close()
print(f'\n[OK] Processed {n} frames in {(time.time()-t0)/60:.1f} min')
print(f'  matte : {matte_path}  ({os.path.getsize(matte_path)/1e6:.1f} MB)')
print(f'  alpha : {alpha_path}  ({os.path.getsize(alpha_path)/1e6:.1f} MB)')

In [ ]:
# Cell 5 — تكوين الخلفية + color match (final QC)
# (العمل الفعلي تم في الخلية السابقة، هنا فقط نتحقق ونولّد thumbnail)
import os, cv2, json

# Extract a thumbnail from the composited video for Telegram preview
cap = cv2.VideoCapture(matte_path)
ok, thumb = cap.read()
cap.release()
if ok:
    thumb_path = os.path.join(out_dir, 'thumbnail.jpg')
    cv2.imwrite(thumb_path, thumb, [cv2.IMWRITE_JPEG_QUALITY, 90])
    print(f'[OK] Thumbnail: {thumb_path}  ({os.path.getsize(thumb_path)/1024:.1f} KB)')

# QC: report file sizes
for p in (matte_path, alpha_path):
    if os.path.exists(p):
        print(f'  {p}  ({os.path.getsize(p)/1e6:.2f} MB)')

In [ ]:
# Cell 6 — رفع النتائج إلى GitHub Release
import os, glob
outputs = [
    matte_path,
    alpha_path,
]
if os.path.exists(os.path.join(out_dir, 'thumbnail.jpg')):
    outputs.append(os.path.join(out_dir, 'thumbnail.jpg'))
outputs = [p for p in outputs if os.path.exists(p)]
print(f'Files to upload: {len(outputs)}')
for p in outputs:
    print(f'  - {p}  ({os.path.getsize(p)/1e6:.2f} MB)')

import os
import json
import subprocess
import urllib.request
import urllib.error

# --- إعدادات الرفع إلى GitHub Release ---
# يتم تمريرها عبر Kaggle Secrets أو متغيرات البيئة
GH_TOKEN = os.environ.get('GH_TOKEN', os.environ.get('GITHUB_TOKEN', ''))
GH_REPO = os.environ.get('GH_REPO', 'mickystudio/micky-infra')
TASK_ID = os.environ.get('KAGGLE_TASK_ID', 'test')
RELEASE_TAG = os.environ.get('RELEASE_TAG', f'task-{TASK_ID}')


def gh_release_api(method: str, path: str, payload: dict | None = None,
                   upload_url: str | None = None, asset_path: str | None = None):
    """Calls the GitHub REST API (or uploads an asset) with auth."""
    headers = {
        'Accept': 'application/vnd.github+json',
        'Authorization': f'Bearer {GH_TOKEN}',
        'X-GitHub-Api-Version': '2022-11-28',
    }
    try:
        if upload_url and asset_path:
            # Upload asset via multipart form
            import mimetypes
            mime = mimetypes.guess_type(asset_path)[0] or 'application/octet-stream'
            headers['Content-Type'] = mime
            with open(asset_path, 'rb') as fh:
                req = urllib.request.Request(upload_url, data=fh.read(), headers=headers, method='POST')
        else:
            data = json.dumps(payload).encode() if payload else None
            base = 'https://api.github.com'
            url = f'{base}{path}' if upload_url is None else upload_url
            req = urllib.request.Request(url, data=data, headers=headers, method=method)
        with urllib.request.urlopen(req, timeout=120) as resp:
            body = resp.read().decode('utf-8', errors='ignore')
            return resp.status, json.loads(body) if body else {}
    except urllib.error.HTTPError as e:
        err_body = e.read().decode('utf-8', errors='ignore')
        print(f'GitHub API error {e.code}: {err_body}')
        return e.code, {}
    except Exception as e:
        print(f'GitHub upload exception: {e}')
        return 0, {}


def ensure_release():
    """Create the release if it doesn't exist, return its upload_url template."""
    # Try fetching first
    status, data = gh_release_api('GET', f'/repos/{GH_REPO}/releases/tags/{RELEASE_TAG}')
    if status == 200:
        upload_url = data.get('upload_url', '').replace('{?name,label}', '')
        return upload_url
    # Create the release
    payload = {
        'tag_name': RELEASE_TAG,
        'name': f'Kaggle task {TASK_ID}',
        'body': 'Auto-uploaded from Kaggle notebook',
        'draft': False,
        'prerelease': False,
    }
    status, data = gh_release_api('POST', f'/repos/{GH_REPO}/releases', payload=payload)
    return data.get('upload_url', '').replace('{?name,label}', '') if data else ''


def upload_asset(upload_url: str, asset_path: str) -> str | None:
    """Uploads a single file to the release. Returns the browser_download_url."""
    name = os.path.basename(asset_path)
    url = f'{upload_url}?name={name}'
    status, data = gh_release_api('POST', '', upload_url=url, asset_path=asset_path)
    if status in (201, 200) and data:
        print(f'  uploaded: {name} -> {data.get("browser_download_url")}')
        return data.get('browser_download_url')
    print(f'  FAILED to upload {name} (status={status})')
    return None


def upload_files_to_release(file_paths, manifest=None):
    """Uploads all files to the GitHub release for this task."""
    if not GH_TOKEN:
        print('WARNING: GH_TOKEN not set. Skipping upload to GitHub Release.')
        print('Files remain available on the Kaggle output at /kaggle/working/output')
        return []
    upload_url = ensure_release()
    if not upload_url:
        print('ERROR: could not create/find GitHub release. Aborting upload.')
        return []
    uploaded = []
    for fp in file_paths:
        if not os.path.exists(fp):
            print(f'  skip (missing): {fp}')
            continue
        url = upload_asset(upload_url, fp)
        if url:
            uploaded.append(url)
    # Optional manifest (JSON describing outputs)
    if manifest:
        manifest_path = '/kaggle/working/manifest.json'
        with open(manifest_path, 'w') as f:
            json.dump(manifest, f, ensure_ascii=False, indent=2)
        upload_asset(upload_url, manifest_path)
    return uploaded


print(f'Release tag: {RELEASE_TAG}')
print(f'Repo:         {GH_REPO}')
print(f'Token set:    {bool(GH_TOKEN)}')

manifest = {
    'task_id': os.environ.get('KAGGLE_TASK_ID', 'test'),
    'input_video': input_video,
    'background': BG_NAME,
    'model': MODEL_VARIANT,
    'frames': n,
    'outputs': [{'file': os.path.basename(p), 'size_mb': round(os.path.getsize(p)/1e6, 2)} for p in outputs],
}
uploaded_urls = upload_files_to_release(outputs, manifest=manifest)
print(f'\n[OK] {len(uploaded_urls)} files uploaded to GitHub Release.')
for u in uploaded_urls:
    print(f'  {u}')

## تم!

ملف `matte_composited.mp4` جاهز ليُمرّر إلى `color_grader.py` (P4-T4) لإضافة التدرّج اللوني السينمائي.